In [ ]:
# ============================================================
# MOIRAI 2.0 — CLEAN ISOLATED ENVIRONMENT
# Latest stable Uni2TS release: 2.0.0
# ============================================================

!rm -rf /content/moirai2_env

!python -m pip install -q virtualenv
!python -m virtualenv /content/moirai2_env

print("\n--- Installing compatible CPU PyTorch ---")
!/content/moirai2_env/bin/pip install --no-cache-dir \
    "torch==2.4.1+cpu" \
    --index-url https://download.pytorch.org/whl/cpu

print("\n--- Installing latest stable Uni2TS ---")
!/content/moirai2_env/bin/pip install --no-cache-dir \
    "uni2ts==2.0.0"

print("\n--- Dependency check ---")
!/content/moirai2_env/bin/pip check

print("\n--- Version / API verification ---")
!/content/moirai2_env/bin/python -c "import torch; import importlib.metadata as m; from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module; print('Torch:', torch.__version__); print('Uni2TS:', m.version('uni2ts')); print('MOIRAI 2 API: OK')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 35.2 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 1025ms
  creator CPython3Posix(dest=/content/moirai2_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator

--- Installing compatible CPU PyTorch ---
Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 260.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 269.5 MB/s  0:00:00
   ━━━━━━

In [ ]:
!/content/moirai2_env/bin/python -c "from uni2ts.model.moirai2 import Moirai2Module; MODEL_ID='Salesforce/moirai-2.0-R-small'; print('Loading:', MODEL_ID); model=Moirai2Module.from_pretrained(MODEL_ID); params=sum(p.numel() for p in model.parameters()); print('MOIRAI 2.0 CHECKPOINT: OK'); print('Model:', MODEL_ID); print('Parameters:', format(params, ',')); print('Patch size:', model.patch_size); print('Quantile levels:', model.quantile_levels)"

Loading: Salesforce/moirai-2.0-R-small



MOIRAI 2.0 CHECKPOINT: OK
Model: Salesforce/moirai-2.0-R-small
Parameters: 11,387,208
Patch size: 16
Quantile levels: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving evaluation_window.csv to evaluation_window.csv


In [ ]:
import pandas as pd
import numpy as np

PATH = "/content/evaluation_window.csv"

df = pd.read_csv(PATH, parse_dates=["timestamp"])

assert len(df) == 120, f"Expected 120 rows, got {len(df)}"
assert (df["split"] == "context").sum() == 96
assert (df["split"] == "ground_truth").sum() == 24
assert df["traffic_speed"].notna().all()
assert (df["traffic_speed"] > 0).all()

context = df[df["split"] == "context"]
gt = df[df["split"] == "ground_truth"]

assert context["timestamp"].iloc[0] == pd.Timestamp("2012-06-04 13:30:00")
assert context["timestamp"].iloc[-1] == pd.Timestamp("2012-06-04 21:25:00")

assert gt["timestamp"].iloc[0] == pd.Timestamp("2012-06-04 21:30:00")
assert gt["timestamp"].iloc[-1] == pd.Timestamp("2012-06-04 23:25:00")

expected_gt = np.array([65.125, 56.625, 64.0, 64.125, 58.0])

assert np.allclose(
    gt["traffic_speed"].to_numpy()[:5],
    expected_gt
)

print("===================================")
print("MOIRAI 2 DATA VALIDATION")
print("===================================")
print("Rows:", len(df))
print("Context:", len(context))
print("Forecast horizon:", len(gt))
print("Context:", context["timestamp"].iloc[0], "→", context["timestamp"].iloc[-1])
print("Ground Truth:", gt["timestamp"].iloc[0], "→", gt["timestamp"].iloc[-1])
print("First 5 GT:", gt["traffic_speed"].to_numpy()[:5])
print()
print("✅ EXACT EVALUATION WINDOW VERIFIED")

MOIRAI 2 DATA VALIDATION
Rows: 120
Context: 96
Forecast horizon: 24
Context: 2012-06-04 13:30:00 → 2012-06-04 21:25:00
Ground Truth: 2012-06-04 21:30:00 → 2012-06-04 23:25:00
First 5 GT: [65.125 56.625 64.    64.125 58.   ]

✅ EXACT EVALUATION WINDOW VERIFIED


In [ ]:
%%writefile /content/run_moirai2.py

import time
import numpy as np
import pandas as pd
import torch

from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# LOCKED EXPERIMENT CONFIG
# ============================================================

MODEL_ID = "Salesforce/moirai-2.0-R-small"
INPUT_PATH = "/content/evaluation_window.csv"
OUTPUT_PATH = "/content/moirai2_predictions.csv"

CONTEXT_LENGTH = 96
FORECAST_HORIZON = 24

np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# 1. LOAD EXACT SAME EVALUATION WINDOW
# ============================================================

df = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])

context_df = df[df["split"] == "context"].copy()
gt_df = df[df["split"] == "ground_truth"].copy()

assert len(context_df) == CONTEXT_LENGTH
assert len(gt_df) == FORECAST_HORIZON

context = context_df["traffic_speed"].to_numpy(dtype=np.float32)
y_true = gt_df["traffic_speed"].to_numpy(dtype=np.float64)

assert context.shape == (96,)
assert y_true.shape == (24,)
assert np.isfinite(context).all()
assert np.isfinite(y_true).all()

print("Input validation: OK")
print("Model will see ONLY:", len(context), "past points")
print("Ground truth kept separate:", len(y_true), "points")

# ============================================================
# 2. LOAD LATEST MOIRAI 2.0 CHECKPOINT
# ============================================================

print("\nLoading:", MODEL_ID)

module = Moirai2Module.from_pretrained(MODEL_ID)

model = Moirai2Forecast(
    module=module,
    prediction_length=FORECAST_HORIZON,
    context_length=CONTEXT_LENGTH,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)

model = model.to("cpu")
model.eval()

print("Moirai 2.0 model: OK")

# ============================================================
# 3. ZERO-SHOT INFERENCE
# ============================================================

start = time.perf_counter()

with torch.no_grad():
    forecast = model.predict(
        [context]
    )

elapsed = time.perf_counter() - start

forecast = np.asarray(forecast)

print("\nRaw forecast shape:", forecast.shape)

# Expected:
# batch x quantiles x horizon
assert forecast.shape[0] == 1
assert forecast.shape[1] == 9
assert forecast.shape[2] == FORECAST_HORIZON

# ============================================================
# 4. USE q=0.5 MEDIAN AS POINT FORECAST
# ============================================================

quantile_levels = list(module.quantile_levels)

assert 0.5 in quantile_levels

median_idx = quantile_levels.index(0.5)

prediction = forecast[
    0,
    median_idx,
    :FORECAST_HORIZON
].astype(np.float64)

assert prediction.shape == (24,)
assert np.isfinite(prediction).all()

# ============================================================
# 5. METRICS
# ============================================================

mae = np.mean(np.abs(y_true - prediction))
rmse = np.sqrt(np.mean((y_true - prediction) ** 2))

# ============================================================
# 6. SAVE POINT + QUANTILE FORECASTS
# ============================================================

result = pd.DataFrame({
    "timestamp": gt_df["timestamp"].to_numpy(),
    "ground_truth": y_true,
    "moirai2_prediction": prediction,
})

for i, q in enumerate(quantile_levels):
    q_name = f"q{int(round(q * 100)):02d}"
    result[q_name] = forecast[0, i, :FORECAST_HORIZON]

result.to_csv(OUTPUT_PATH, index=False)

# ============================================================
# 7. FINAL REPORT
# ============================================================

print("\n======================================")
print("MOIRAI 2.0 ZERO-SHOT RESULT")
print("======================================")

print("Model:", MODEL_ID)
print("Sensor: 773062")
print("Variable: traffic speed")
print("Context:", CONTEXT_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Sampling interval: 5 minutes")
print("Training / fine-tuning: NONE")
print("Point forecast: q=0.5 median")

print("\nFirst 5 predictions:")
print(prediction[:5])

print("\nFirst 5 ground truth:")
print(y_true[:5])

print("\n====================")
print("METRICS")
print("====================")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

print(f"\nInference time (CPU): {elapsed:.4f} sec")

print("\nSaved:", OUTPUT_PATH)
print("Rows:", len(result))

print("\n✅ MOIRAI 2.0 ZERO-SHOT INFERENCE COMPLETED")

Writing /content/run_moirai2.py


In [ ]:
!/content/moirai2_env/bin/python /content/run_moirai2.py

Input validation: OK
Model will see ONLY: 96 past points
Ground truth kept separate: 24 points

Loading: Salesforce/moirai-2.0-R-small
Moirai 2.0 model: OK

Raw forecast shape: (1, 9, 24)

MOIRAI 2.0 ZERO-SHOT RESULT
Model: Salesforce/moirai-2.0-R-small
Sensor: 773062
Variable: traffic speed
Context: 96
Forecast horizon: 24
Sampling interval: 5 minutes
Training / fine-tuning: NONE
Point forecast: q=0.5 median

First 5 predictions:
[64.81654358 64.95730591 65.03863525 65.11444855 64.97059631]

First 5 ground truth:
[65.125 56.625 64.    64.125 58.   ]

METRICS
MAE : 1.6552
RMSE: 2.7551

Inference time (CPU): 0.1597 sec

Saved: /content/moirai2_predictions.csv
Rows: 24

✅ MOIRAI 2.0 ZERO-SHOT INFERENCE COMPLETED


In [ ]:
from google.colab import files
files.download("/content/moirai2_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>